# AgriResolve AI - Exploratory Data Analysis (EDA)
### Phase 7: Data Exploration, Pattern Recognition & Rule Threshold Justification

**Objective**:
1. Analyze rainfall deviations against 10-year IMD baseline (Real Data).
2. Observe Sentinel-2 10m NDVI vegetation health trajectories and drop events (Real Data).
3. Inspect reported farmer distress clusters across crops and categories (Synthetic Data).
4. Audit field inspection recency gaps and data reliability indices (Synthetic & Derived Data).
5. Formulate transparent mathematical thresholds for the subsequent Routine and Reported Priority Engines.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Paths
clean_dir = Path('../data/cleaned')
derived_dir = Path('../data/derived')
synthetic_dir = Path('../data/synthetic')

# Read Cleaned & Derived Excel files
df_villages = pd.read_excel(clean_dir / 'villages_cleaned.xlsx')
df_rain = pd.read_excel(derived_dir / 'rainfall_anomaly_derived.xlsx')
df_ndvi = pd.read_excel(derived_dir / 'ndvi_trends_derived.xlsx')
df_complaints = pd.read_excel(clean_dir / 'complaints_cleaned.xlsx')
df_inspections = pd.read_excel(clean_dir / 'inspections_cleaned.xlsx')
df_reliability = pd.read_excel(derived_dir / 'village_reliability_derived.xlsx')
print('Data successfully loaded from Excel files.')

## 1. Rainfall Anomaly vs. Historical Normal Baseline (Real IMD/KSNDMC Data)
We calculate the percentage anomaly as:  
$$\text{Anomaly (\%)} = \frac{\text{Actual} - \text{Normal}}{\text{Normal}} \times 100$$

In [2]:
fig_rain = px.bar(
    df_rain,
    x='observation_date',
    y='rainfall_anomaly_percent',
    color='rainfall_status',
    facet_col='village_name',
    facet_col_wrap=3,
    title='Rainfall Anomaly (%) vs 10-Year IMD Normal (Maddur Cluster)',
    color_discrete_map={
        'Excess': '#2ca02c',
        'Normal': '#1f77b4',
        'Deficit': '#ff7f0e',
        'Severe Deficit': '#d62728'
    }
)
fig_rain.update_layout(height=700, template='plotly_white')
fig_rain.show()

## 2. Sentinel-2 10m NDVI Vegetation Trajectory (Real Satellite Observations)
Identifying mid-season vegetation dips indicative of moisture stress or pest defoliation.

In [3]:
fig_ndvi = px.line(
    df_ndvi,
    x='observation_date',
    y='mean_ndvi',
    color='village_name',
    markers=True,
    title='Sentinel-2 MSI 10m NDVI Trajectory (Kharif - Rabi 2024)',
    labels={'mean_ndvi': 'Mean Agricultural NDVI', 'observation_date': 'Observation Date'}
)
fig_ndvi.add_hline(y=0.30, line_dash='dash', line_color='red', annotation_text='Stress Level (<0.30)')
fig_ndvi.update_layout(height=500, template='plotly_white')
fig_ndvi.show()

## 3. Reported Farmer Complaints Distribution (Flagged Synthetic Data)
*Note: All complaints are statistically generated benchmark records.*

In [4]:
fig_comp = px.sunburst(
    df_complaints,
    path=['village_name', 'crop_affected', 'issue_category'],
    values='affected_area_hectares',
    color='urgency_level',
    title='Farmer Distress by Village, Crop & Issue (Area in Hectares)',
    color_discrete_map={'Low': '#2ca02c', 'Medium': '#ffbb78', 'High': '#ff7f0e', 'Critical': '#d62728'}
)
fig_comp.update_layout(height=600)
fig_comp.show()

## 4. Inspection Recency Gap & Data Reliability Index

In [5]:
fig_insp = px.bar(
    df_inspections.sort_values('days_since_last_inspection', ascending=False),
    x='days_since_last_inspection',
    y='village_name',
    orientation='h',
    color='days_since_last_inspection',
    color_continuous_scale='Reds',
    title='Days Since Last Field Inspection by Village',
    labels={'days_since_last_inspection': 'Days Elapsed', 'village_name': 'Village'}
)
fig_insp.add_vline(x=60, line_dash='dash', line_color='red', annotation_text='Overdue Threshold (60d)')
fig_insp.update_layout(height=400, template='plotly_white')
fig_insp.show()